In [51]:
# Install required packages
!pip install nltk python-Levenshtein matplotlib torch


In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
import Levenshtein
from collections import Counter
import math
import random

# Download required NLTK data

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [53]:
def encode_sentence(sentence, token2id, is_urdu=True):
    """Encode a sentence using greedy longest-match subword tokenization."""
    tokens = []

    if is_urdu:
        words = ["_" + w for w in sentence.split()]
    else:
        words = [w + "_" for w in sentence.split()]

    for w in words:
        i = 0
        while i < len(w):
            subword = None
            for j in range(len(w), i, -1):
                piece = w[i:j]
                if piece in token2id:
                    subword = piece
                    break
            if subword is None:
                tokens.append(token2id["<unk>"])
                i += 1
            else:
                tokens.append(token2id[subword])
                i += len(subword)

    return [token2id["<sos>"]] + tokens + [token2id["<eos>"]]



class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
        self.src_data = []
        self.tgt_data = []

        for src, tgt in zip(src_sentences, tgt_sentences):
            src_tokens = encode_sentence(src, src_vocab, is_urdu=True)
            tgt_tokens = encode_sentence(tgt, tgt_vocab, is_urdu=False)

            self.src_data.append(torch.tensor(src_tokens))
            self.tgt_data.append(torch.tensor(tgt_tokens))

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    # lengths
    max_src_len = max(len(src) for src in src_batch)
    max_tgt_len = max(len(tgt) for tgt in tgt_batch)

    # we force both to same max length
    max_len = max(max_src_len, max_tgt_len)

    # pad each sequence with 0 up to max_len
    src_padded = [F.pad(src, (0, max_len - len(src)), value=0) for src in src_batch]
    tgt_padded = [F.pad(tgt, (0, max_len - len(tgt)), value=0) for tgt in tgt_batch]

    # stack into batch tensors
    return torch.stack(src_padded), torch.stack(tgt_padded)



In [54]:


class Encoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=256, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                             dropout=dropout, bidirectional=True, batch_first=True)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, (h, c) = self.bilstm(embedded)
        return output, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=512, hidden_dim=256, num_layers=3,
                 output_vocab_size=512, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Project encoder states to decoder dimensions
        self.h_projection = nn.Linear(256, hidden_dim)  # 256 from bidirectional encoder
        self.c_projection = nn.Linear(256, hidden_dim)

    def forward(self, x, encoder_states=None):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        if encoder_states is not None:
            h_enc, c_enc = encoder_states
            # Convert bidirectional encoder states to decoder format
            # h_enc: [num_layers*2, batch, hidden_dim] -> [num_layers, batch, hidden_dim*2]
            batch_size = h_enc.size(1)
            h_enc = h_enc.view(2, 2, batch_size, -1)  # [directions, layers, batch, hidden]
            c_enc = c_enc.view(2, 2, batch_size, -1)

            # Concatenate forward and backward states
            h_enc = torch.cat([h_enc[0], h_enc[1]], dim=-1)  # [layers, batch, hidden*2]
            c_enc = torch.cat([c_enc[0], c_enc[1]], dim=-1)

            # Project to decoder dimensions and repeat for all decoder layers
            h_init = self.h_projection(h_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)
            c_init = self.c_projection(c_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)

            initial_state = (h_init, c_init)
        else:
            initial_state = None

        output, _ = self.lstm(embedded, initial_state)
        output = self.linear(output)
        return output

class Seq2SeqModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, src):
        # Encoder processes source
        enc_output, enc_states = self.encoder(src)

        # Decoder processes same source sequence (as per your requirement)
        dec_output = self.decoder(src, enc_states)

        return dec_output

In [55]:
def load_data_and_vocab():
    """Load data and vocabularies from Google Drive"""
    base_path = "/content/drive/MyDrive/Model4/"

    # Load source and target sentences
    with open(base_path + "src_normalized.txt", 'r', encoding='utf-8') as f:
        src_sentences = [line.strip() for line in f]

    with open(base_path + "tgt_normalized.txt", 'r', encoding='utf-8') as f:
        tgt_sentences = [line.strip() for line in f]

    # Load vocabularies
    with open(base_path + "vocab_Urdu.json", 'r', encoding='utf-8') as f:
        urdu_vocab = json.load(f)

    with open(base_path + "vocab_Roman.json", 'r', encoding='utf-8') as f:
        roman_vocab = json.load(f)

    return src_sentences, tgt_sentences, urdu_vocab, roman_vocab

def create_datasets(src_sentences, tgt_sentences, urdu_vocab, roman_vocab):
    """Create train/val/test splits"""
    total_size = len(src_sentences)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)

    # Shuffle data
    indices = list(range(total_size))
    random.shuffle(indices)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create datasets
    train_src = [src_sentences[i] for i in train_indices]
    train_tgt = [tgt_sentences[i] for i in train_indices]

    val_src = [src_sentences[i] for i in val_indices]
    val_tgt = [tgt_sentences[i] for i in val_indices]

    test_src = [src_sentences[i] for i in test_indices]
    test_tgt = [tgt_sentences[i] for i in test_indices]

    train_dataset = TranslationDataset(train_src, train_tgt, urdu_vocab, roman_vocab)
    val_dataset = TranslationDataset(val_src, val_tgt, urdu_vocab, roman_vocab)
    test_dataset = TranslationDataset(test_src, test_tgt, urdu_vocab, roman_vocab)

    return train_dataset, val_dataset, test_dataset

def calculate_perplexity(loss):
    """Calculate perplexity from loss"""
    return math.exp(loss)

def decode_tokens(tokens, id2token):
    """Convert token IDs back to text"""
    words = []
    for token_id in tokens:
        if token_id in [0, 1, 2]:  # pad, sos, eos
            continue
        words.append(id2token.get(token_id, '<unk>'))
    return ' '.join(words)

def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)

def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate"""
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    return Levenshtein.distance(reference, hypothesis) / len(reference)

def calculate_edit_distance(reference, hypothesis):
    """Calculate Levenshtein distance"""
    return Levenshtein.distance(reference, hypothesis)


In [56]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, roman_vocab):
    id2roman = {v: k for k, v in roman_vocab.items()}

    for epoch in range(epochs):
        # ---- TRAINING ----
        model.train()
        total_loss = 0
        correct, total = 0, 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad()
            output = model(src)  # (batch, seq_len, vocab_size)

            # Flatten for CE Loss
            output_flat = output.reshape(-1, output.size(-1))
            tgt_flat = tgt.reshape(-1)

            loss = criterion(output_flat, tgt_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            # Accuracy (token-level)
            predictions = torch.argmax(output, dim=-1)
            correct += (predictions == tgt).sum().item()
            total += tgt.numel()

        avg_train_loss = total_loss / len(train_loader)
        train_perplexity = calculate_perplexity(avg_train_loss)
        train_accuracy = correct / total

        # ---- VALIDATION ----
        model.eval()
        val_loss, val_bleu, val_cer, val_edit, val_correct, val_total = 0, 0, 0, 0, 0, 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                output = model(src)

                # Loss
                output_flat = output.reshape(-1, output.size(-1))
                tgt_flat = tgt.reshape(-1)
                loss = criterion(output_flat, tgt_flat)
                val_loss += loss.item()

                # Predictions
                predictions = torch.argmax(output, dim=-1)
                val_correct += (predictions == tgt).sum().item()
                val_total += tgt.numel()

                for i in range(src.size(0)):
                    pred_tokens = predictions[i].cpu().numpy()
                    tgt_tokens = tgt[i].cpu().numpy()

                    pred_text = decode_tokens(pred_tokens, id2roman)
                    ref_text = decode_tokens(tgt_tokens, id2roman)

                    val_bleu += calculate_bleu(ref_text, pred_text)
                    val_cer += calculate_cer(ref_text, pred_text)
                    val_edit += calculate_edit_distance(ref_text, pred_text)

        # Averages
        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = calculate_perplexity(avg_val_loss)
        avg_val_bleu = val_bleu / len(val_loader.dataset)
        avg_val_cer = val_cer / len(val_loader.dataset)
        avg_val_edit = val_edit / len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # ---- RESULTS ----
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}, Perplexity: {train_perplexity:.4f}, Accuracy: {train_accuracy:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}, "
              f"Accuracy: {val_accuracy:.4f}, BLEU: {avg_val_bleu:.4f}, CER: {avg_val_cer:.4f}, "
              f"Edit Dist: {avg_val_edit:.2f}")




def evaluate_model(model, test_loader, roman_vocab, device):
    """Evaluate model with metrics"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}

    total_bleu = 0
    total_cer = 0
    total_edit_dist = 0
    total_loss = 0
    count = 0

    criterion = nn.CrossEntropyLoss(ignore_index=0)

    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)

            output = model(src)

            # Calculate loss
            loss_output_flat = output.reshape(-1, output.size(-1))
            target_flat = tgt.reshape(-1)
            loss = criterion(loss_output_flat, target_flat)
            total_loss += loss.item()

            predictions = torch.argmax(output, dim=-1)

            for i in range(src.size(0)):
                pred_tokens = predictions[i].cpu().numpy()
                tgt_tokens = tgt[i].cpu().numpy()

                pred_text = decode_tokens(pred_tokens, id2roman)
                ref_text = decode_tokens(tgt_tokens, id2roman)

                total_bleu += calculate_bleu(ref_text, pred_text)
                total_cer += calculate_cer(ref_text, pred_text)
                total_edit_dist += calculate_edit_distance(ref_text, pred_text)
                count += 1

    avg_loss = total_loss / len(test_loader)
    avg_bleu = total_bleu / count
    avg_cer = total_cer / count
    avg_edit_dist = total_edit_dist / count

    print(f"Test Loss: {avg_loss:.4f}, Perplexity: {calculate_perplexity(avg_loss):.4f}")
    print(f"BLEU Score: {avg_bleu:.4f}")
    print(f"Character Error Rate: {avg_cer:.4f}")
    print(f"Average Edit Distance: {avg_edit_dist:.2f}")

    return avg_bleu, avg_cer, avg_edit_dist

def show_examples(model, test_dataset, urdu_vocab, roman_vocab, device, num_examples=5):
    """Show translation examples"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}
    id2urdu = {v: k for k, v in urdu_vocab.items()}

    indices = random.sample(range(len(test_dataset)), num_examples)

    with torch.no_grad():
        for idx in indices:
            src, tgt = test_dataset[idx]
            src_batch = src.unsqueeze(0).to(device)

            output = model(src_batch)
            prediction = torch.argmax(output, dim=-1).squeeze(0)

            src_text = decode_tokens(src.numpy(), id2urdu)
            tgt_text = decode_tokens(tgt.numpy(), id2roman)
            pred_text = decode_tokens(prediction.cpu().numpy(), id2roman)

            print(f"Source (Urdu): {src_text}")
            print(f"Target (Roman): {tgt_text}")
            print(f"Prediction: {pred_text}")

            # Calculate metrics for this example
            bleu = calculate_bleu(tgt_text, pred_text)
            cer = calculate_cer(tgt_text, pred_text)
            edit_dist = calculate_edit_distance(tgt_text, pred_text)

            print(f"BLEU: {bleu:.3f}, CER: {cer:.3f}, Edit Dist: {edit_dist}")
            print("-" * 50)


In [57]:
def test_metrics_sanity_check():
    """Sanity check for evaluation metrics"""
    print("Testing evaluation metrics...")

    # Test BLEU
    ref = "yeh ek test sentence hai"
    hyp = "yeh ek test sentence hai"
    print(f"BLEU (identical): {calculate_bleu(ref, hyp):.4f}")

    hyp = "yeh test sentence hai"
    print(f"BLEU (missing word): {calculate_bleu(ref, hyp):.4f}")

    # Test CER
    ref = "hello world"
    hyp = "hello world"
    print(f"CER (identical): {calculate_cer(ref, hyp):.4f}")

    hyp = "helo wrold"
    print(f"CER (2 errors): {calculate_cer(ref, hyp):.4f}")

    # Test Edit Distance
    print(f"Edit distance (identical): {calculate_edit_distance('hello', 'hello')}")
    print(f"Edit distance (1 substitution): {calculate_edit_distance('hello', 'hallo')}")


In [58]:
test_metrics_sanity_check()

Testing evaluation metrics...
BLEU (identical): 1.0000
BLEU (missing word): 0.3611
CER (identical): 0.0000
CER (2 errors): 0.2727
Edit distance (identical): 0
Edit distance (1 substitution): 1


In [59]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [60]:
# Load data
src_sentences, tgt_sentences, urdu_vocab, roman_vocab = load_data_and_vocab()


In [61]:
# Create datasets
train_dataset, val_dataset, test_dataset = create_datasets(
    src_sentences, tgt_sentences, urdu_vocab, roman_vocab)


In [62]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [63]:

# Initialize model
model = Seq2SeqModel().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


In [64]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _تجھ _سے _ا ل فت _ن ب اہ تا _ہو ں _میں
Target (Roman): tujh_ se_ ul fat_ ni ba h ta _ huun_ main_
Prediction: tire_ tire_ o nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 1.595, Edit Dist: 67
--------------------------------------------------
Source (Urdu): _بو را _گیا _ہے _من ہ _سے _کیوں _کھل تا _نہیں _د ھو اں
Target (Roman): ba u ra _ gaya_ hai_ munh_ se_ kyuun_ kh ul ta _ nahin_ dhu an _
Prediction: r-e- r-e- r-e- aaj_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 1.141, Edit Dist: 73
--------------------------------------------------
Source (Urdu): _مگر _ش ہ زا د ۂ _گل فا م _پر _ش ید ا _نہیں _ہوتی ں
Target (Roman): magar_ sh ah za da-e- gu l fa m_ pa r_ sh a i da _ nahin_ ho ti n_
Prediction: duur_ duur_ duur_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 1.409, Edit

In [65]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
print("Starting training...")

# Step 1: Train encoder only (freeze decoder)
print("\nStep 1: Training encoder (10 epochs)")
for param in model.decoder.parameters():
    param.requires_grad = False

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

train_model(model, train_loader, val_loader, criterion, optimizer, 10, device, roman_vocab)


Starting training...

Step 1: Training encoder (10 epochs)

Epoch 1/10
Train Loss: 6.0455, Perplexity: 422.2041, Accuracy: 0.0369
Val Loss:   6.0276, Perplexity: 414.7065, Accuracy: 0.0367, BLEU: 0.0004, CER: 1.9048, Edit Dist: 101.41

Epoch 2/10
Train Loss: 6.0238, Perplexity: 413.1532, Accuracy: 0.0373
Val Loss:   6.0162, Perplexity: 410.0240, Accuracy: 0.0367, BLEU: 0.0005, CER: 1.8786, Edit Dist: 100.08

Epoch 3/10
Train Loss: 6.0128, Perplexity: 408.6454, Accuracy: 0.0373
Val Loss:   6.0075, Perplexity: 406.4637, Accuracy: 0.0368, BLEU: 0.0007, CER: 1.8907, Edit Dist: 100.73

Epoch 4/10
Train Loss: 6.0062, Perplexity: 405.9500, Accuracy: 0.0374
Val Loss:   6.0025, Perplexity: 404.4418, Accuracy: 0.0368, BLEU: 0.0008, CER: 1.9023, Edit Dist: 101.34

Epoch 5/10
Train Loss: 6.0016, Perplexity: 404.0669, Accuracy: 0.0374
Val Loss:   5.9989, Perplexity: 403.0040, Accuracy: 0.0369, BLEU: 0.0009, CER: 1.8981, Edit Dist: 101.12

Epoch 6/10
Train Loss: 5.9980, Perplexity: 402.6100, Accurac

In [67]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ل ی لی ٰ _ن ے _ن یا _جن م _ل یا _ہے
Target (Roman): la il a_ ne_ na ya_ ja nam_ liya_ hai_
Prediction: aaj_ aaj_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 0.842, Edit Dist: 32
--------------------------------------------------
Source (Urdu): _سو _کا ہ ے _کو _اپنی _تو _جو گی _کی _سی _پ ھی ری _ہے
Target (Roman): so _ ka ah e_ ko_ apni_ tu_ jo gi_ ki_ si _ ph er i_ hai_
Prediction: dat_ aaj_ aaj_ aaj_ aaj_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 0.772, Edit Dist: 44
--------------------------------------------------
Source (Urdu): _تم ا شا _کر د نی _ہے _ل ط ف _زخم _انت ظ ار _اے _دل
Target (Roman): tama sh a_ ka r da ni _ hai_ lu t f-e- zakh m-e- inti za r_ ai_ dil_
Prediction: nu aaj_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_ nikle_
BLEU: 0.000, CER: 0.824, Edit Dist: 56
--------------------------------------------------
Source (Urdu): _ا شک _ا ف شا نی _بھی _میری _گو ہر _ا ف شا نی _ہوئی
Targe

In [68]:
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (15 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 15, device,roman_vocab)



Step 2: Training decoder (15 epochs)

Epoch 1/15
Train Loss: 4.4443, Perplexity: 85.1393, Accuracy: 0.1276
Val Loss:   3.9215, Perplexity: 50.4762, Accuracy: 0.1630, BLEU: 0.0389, CER: 0.5069, Edit Dist: 28.38

Epoch 2/15
Train Loss: 3.5608, Perplexity: 35.1896, Accuracy: 0.1892
Val Loss:   3.2041, Perplexity: 24.6334, Accuracy: 0.2103, BLEU: 0.0860, CER: 0.4325, Edit Dist: 24.26

Epoch 3/15
Train Loss: 3.0362, Perplexity: 20.8260, Accuracy: 0.2247
Val Loss:   2.8177, Perplexity: 16.7391, Accuracy: 0.2405, BLEU: 0.1362, CER: 0.3748, Edit Dist: 21.13

Epoch 4/15
Train Loss: 2.7246, Perplexity: 15.2500, Accuracy: 0.2512
Val Loss:   2.5807, Perplexity: 13.2066, Accuracy: 0.2634, BLEU: 0.1753, CER: 0.3427, Edit Dist: 19.33

Epoch 5/15
Train Loss: 2.5171, Perplexity: 12.3922, Accuracy: 0.2729
Val Loss:   2.4345, Perplexity: 11.4104, Accuracy: 0.2782, BLEU: 0.1999, CER: 0.3303, Edit Dist: 18.68

Epoch 6/15
Train Loss: 2.3714, Perplexity: 10.7122, Accuracy: 0.2875
Val Loss:   2.3439, Perplex

In [69]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اس _پہ _بھی _کچھ _نہ _پوچھ و _کی ا _کچھ _ہو ں
Target (Roman): us_ pe_ bhi_ kuchh_ na_ pu chho _ kya_ kuchh_ huun_
Prediction: us_ pa bhi_ kuchh_ na_ puchh_ chho _ kya_ kuchh_ kuchh_
BLEU: 0.351, CER: 0.196, Edit Dist: 10
--------------------------------------------------
Source (Urdu): _کسی _کے _صب ر _ن ے _بے _صب ر _دی ا _سب _کو
Target (Roman): ki si _ ke_ sa b r_ ne_ be- sa b r_ diya_ sa b_ ko_
Prediction: ki si _ ke_ sa b r_ ne_ ne_ sa b b r_ sa
BLEU: 0.525, CER: 0.294, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _ک ٹ ے _غم _ہی ، _ن ک لے _جو _د م _مر ا _مجھ ے _اپنی _زندگی _با ر _ہے
Target (Roman): ka te_ gha m_ hi i , _ nikle_ jo_ dam_ mi ra _ mujhe_ apni_ zindagi_ ba ar_ hai_
Prediction: ka t_ e_ gha hi_ , _ nik a jo_ da ma r _ mujhe_
BLEU: 0.045, CER: 0.512, Edit Dist: 41
--------------------------------------------------
Source (Urdu): _پر _اک _سک و ن _تھا _ہم _کو _فر یب _ک ھا نے _میں
Target (Roman): pa r_ 

In [70]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 1.7239, Perplexity: 5.6064, Accuracy: 0.3730
Val Loss:   2.0387, Perplexity: 7.6802, Accuracy: 0.3463, BLEU: 0.3374, CER: 0.2691, Edit Dist: 14.92

Epoch 2/10
Train Loss: 1.6843, Perplexity: 5.3887, Accuracy: 0.3805
Val Loss:   2.1070, Perplexity: 8.2236, Accuracy: 0.3434, BLEU: 0.3368, CER: 0.2771, Edit Dist: 15.25

Epoch 3/10
Train Loss: 1.6584, Perplexity: 5.2511, Accuracy: 0.3843
Val Loss:   2.0410, Perplexity: 7.6985, Accuracy: 0.3517, BLEU: 0.3531, CER: 0.2536, Edit Dist: 14.17

Epoch 4/10
Train Loss: 1.6311, Perplexity: 5.1096, Accuracy: 0.3857
Val Loss:   2.0160, Perplexity: 7.5083, Accuracy: 0.3536, BLEU: 0.3566, CER: 0.2516, Edit Dist: 14.09

Epoch 5/10
Train Loss: 1.6076, Perplexity: 4.9906, Accuracy: 0.3900
Val Loss:   2.0176, Perplexity: 7.5201, Accuracy: 0.3519, BLEU: 0.3598, CER: 0.2497, Edit Dist: 13.98

Epoch 6/10
Train Loss: 1.5759, Perplexity: 4.8349, Accuracy: 0.3960
Val Loss:   2.0219, Perplexity: 7.5525

In [71]:
# Step 3: Train full model
print("\nStep 3: Training full model (20 epochs)")
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0005)
train_model(model, train_loader, val_loader, criterion, optimizer, 20, device,roman_vocab)



Step 3: Training full model (10 epochs)

Epoch 1/20
Train Loss: 1.4087, Perplexity: 4.0905, Accuracy: 0.4209
Val Loss:   1.9900, Perplexity: 7.3155, Accuracy: 0.3685, BLEU: 0.3905, CER: 0.2402, Edit Dist: 13.32

Epoch 2/20
Train Loss: 1.3603, Perplexity: 3.8973, Accuracy: 0.4284
Val Loss:   1.9979, Perplexity: 7.3733, Accuracy: 0.3696, BLEU: 0.3935, CER: 0.2427, Edit Dist: 13.43

Epoch 3/20
Train Loss: 1.3287, Perplexity: 3.7761, Accuracy: 0.4330
Val Loss:   1.9971, Perplexity: 7.3674, Accuracy: 0.3714, BLEU: 0.3970, CER: 0.2406, Edit Dist: 13.32

Epoch 4/20
Train Loss: 1.3041, Perplexity: 3.6844, Accuracy: 0.4373
Val Loss:   1.9904, Perplexity: 7.3188, Accuracy: 0.3742, BLEU: 0.4025, CER: 0.2369, Edit Dist: 13.17

Epoch 5/20
Train Loss: 1.2822, Perplexity: 3.6046, Accuracy: 0.4420
Val Loss:   1.9777, Perplexity: 7.2264, Accuracy: 0.3757, BLEU: 0.4074, CER: 0.2305, Edit Dist: 12.82

Epoch 6/20
Train Loss: 1.2642, Perplexity: 3.5401, Accuracy: 0.4466
Val Loss:   1.9910, Perplexity: 7.3

In [72]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _کہیں _ا ند ھی رے _سے _م ان وس _ہو _نہ _جائے _ا د ب
Target (Roman): ka hi n_ an dh er e_ se_ ma nu s_ ho_ na_ jaae_ a da b_
Prediction: ka hi n_ dh er e_ se_ ma ma su _ ho_ jaae_ jaae_ un da
BLEU: 0.311, CER: 0.273, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _نہ _ٹھ کا نا _ہے _جگر _کا _نہ _ٹھ کا نا _دل _کا
Target (Roman): na_ t hi ka na_ hai_ jigar_ ka_ na_ t hi ka na_ dil_ ka_
Prediction: na_ t hi ka na_ hai_ jigar_ ka_ na_ t hi ka na_ dil_
BLEU: 0.931, CER: 0.071, Edit Dist: 4
--------------------------------------------------
Source (Urdu): _ہے _ہے _عر ق _عر ق _وہ _ت ن _ناز نی ں _رہے
Target (Roman): hai_ hai_ ar a q_ ar a q_ vo_ ta n -e- naz ni n_ ra he_
Prediction: hai_ hai_ ar q q ar q q_ ta n -e- naz naz n_
BLEU: 0.227, CER: 0.291, Edit Dist: 16
--------------------------------------------------
Source (Urdu): _ا د ب _ہے _اور _یہی _ک شم ک ش _تو _کی ا _کی ج ے
Target (Roman): a da b_ hai_ aur_ ya hi_ ka sh m

In [75]:
# Step 4: Fine-tune with low learning rate and decay
print("\nStep 4: Fine-tuning with learning rate decay (10 epochs)")
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

for epoch in range(10):
    train_model(model, train_loader, val_loader, criterion, optimizer, 1, device, roman_vocab)
    scheduler.step()
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")



Step 4: Fine-tuning with learning rate decay (10 epochs)

Epoch 1/1
Train Loss: 1.0204, Perplexity: 2.7743, Accuracy: 0.4811
Val Loss:   2.0721, Perplexity: 7.9416, Accuracy: 0.3854, BLEU: 0.4285, CER: 0.2203, Edit Dist: 12.19
Learning rate: 0.000090

Epoch 1/1
Train Loss: 0.9901, Perplexity: 2.6915, Accuracy: 0.4871
Val Loss:   2.0655, Perplexity: 7.8896, Accuracy: 0.3869, BLEU: 0.4328, CER: 0.2208, Edit Dist: 12.20
Learning rate: 0.000081

Epoch 1/1
Train Loss: 0.9664, Perplexity: 2.6284, Accuracy: 0.4903
Val Loss:   2.0787, Perplexity: 7.9938, Accuracy: 0.3875, BLEU: 0.4337, CER: 0.2209, Edit Dist: 12.19
Learning rate: 0.000073

Epoch 1/1
Train Loss: 0.9606, Perplexity: 2.6133, Accuracy: 0.4902
Val Loss:   2.0795, Perplexity: 8.0003, Accuracy: 0.3878, BLEU: 0.4340, CER: 0.2202, Edit Dist: 12.16
Learning rate: 0.000066

Epoch 1/1
Train Loss: 0.9462, Perplexity: 2.5758, Accuracy: 0.4909
Val Loss:   2.0697, Perplexity: 7.9221, Accuracy: 0.3883, BLEU: 0.4353, CER: 0.2193, Edit Dist: 12

In [76]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.0001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 0.9148, Perplexity: 2.4962, Accuracy: 0.4993
Val Loss:   2.0889, Perplexity: 8.0761, Accuracy: 0.3898, BLEU: 0.4359, CER: 0.2213, Edit Dist: 12.18

Epoch 2/10
Train Loss: 0.9056, Perplexity: 2.4734, Accuracy: 0.4995
Val Loss:   2.1026, Perplexity: 8.1878, Accuracy: 0.3893, BLEU: 0.4350, CER: 0.2223, Edit Dist: 12.22

Epoch 3/10
Train Loss: 0.9074, Perplexity: 2.4779, Accuracy: 0.5008
Val Loss:   2.0906, Perplexity: 8.0898, Accuracy: 0.3902, BLEU: 0.4357, CER: 0.2198, Edit Dist: 12.12

Epoch 4/10
Train Loss: 0.8933, Perplexity: 2.4431, Accuracy: 0.5021
Val Loss:   2.0960, Perplexity: 8.1335, Accuracy: 0.3907, BLEU: 0.4369, CER: 0.2200, Edit Dist: 12.11

Epoch 5/10
Train Loss: 0.8864, Perplexity: 2.4264, Accuracy: 0.5031
Val Loss:   2.0962, Perplexity: 8.1350, Accuracy: 0.3906, BLEU: 0.4360, CER: 0.2202, Edit Dist: 12.13

Epoch 6/10
Train Loss: 0.8840, Perplexity: 2.4206, Accuracy: 0.5055
Val Loss:   2.0973, Perplexity: 8.1443

In [77]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اور _کچھ _دن _ابھی _اور وں _کو _س تا یا _جائے
Target (Roman): aur_ kuchh_ di n_ ab hi_ a u ro n_ ko_ sa ta ya_ jaae_
Prediction: aur_ kuchh_ di n_ ab hi_ aur_ ro ro n_ ko_ sa
BLEU: 0.494, CER: 0.315, Edit Dist: 17
--------------------------------------------------
Source (Urdu): _کوئی _مے _دے _ی ا _نہ _دے _ہم _ر ند _بے _پر وا _ہیں _آپ
Target (Roman): koi_ mai _ de _ ya_ na_ de _ ham_ ri n d-e- be- pa r va _ hain_ aap_
Prediction: koi_ mai- _ _ ya_ na_ de _ ham_ ri n d-e- be- pa r va
BLEU: 0.639, CER: 0.250, Edit Dist: 17
--------------------------------------------------
Source (Urdu): _غالبؔ _کچھ _اپنی _س ع ی _سے _ل ہ نا _نہیں _مجھ ے
Target (Roman): 'ghalib'_ kuchh_ apni_ sa i_ se_ la h na_ nahin_ mujhe_
Prediction: 'ghalib'_ kuchh_ apni_ sa o i_ sh la kh _ _
BLEU: 0.260, CER: 0.327, Edit Dist: 18
--------------------------------------------------
Source (Urdu): _ذ کر _ہے _کچھ _گل ہ _نہیں _بات _ہے _ن یش ت ر _نہیں
Target (Roman): zi kr_ hai_ kuchh

In [78]:
# Evaluation
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)



Evaluation Results:
Test Loss: 2.0432, Perplexity: 7.7153
BLEU Score: 0.4463
Character Error Rate: 0.2163
Average Edit Distance: 11.84


(0.4463302625266382, 0.21634270227847924, 11.841163310961969)

In [80]:
save_path = "/content/drive/MyDrive/Model4/urdu_roman_nmt_model.pth"
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'urdu_vocab': urdu_vocab,
    'roman_vocab': roman_vocab
}, save_path)
print("Model saved as 'urdu_roman_nmt_model.pth'")



Model saved as 'urdu_roman_nmt_model.pth'
